# Chapter 9 §9.2: Invoice Entity Extraction

Partial-credit metrics, field-level majority voting, and GEPA improve structured extraction across varied invoice layouts.


In [ ]:
%pip install -r ../requirements.txt -q


## Runtime setup

Cells tagged `chapter-example` contain the chapter's core examples. Supporting cells provide imports, fixtures, or safe local stand-ins for external services.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import dspy

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in ../.env before running this notebook.")

lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)


## Extraction signature


In [ ]:
class InvoiceExtraction(dspy.Signature):
    """Extract structured invoice fields from free-form invoice text."""

    text: str = dspy.InputField(desc="Raw invoice text")
    rationale: str = dspy.OutputField(
        desc="Brief reasoning about detected fields"
    )
    company: str = dspy.OutputField(desc="Company name (issuer/seller)")
    billed_to: str = dspy.OutputField(desc="Customer/recipient name")
    invoice_number: str = dspy.OutputField(desc="Invoice number/ID")
    invoice_date: str = dspy.OutputField(desc="Invoice issue date")
    total_amount: str = dspy.OutputField(
        desc="Total amount with currency"
    )
    bank_name: str = dspy.OutputField(desc="Bank name for payment")
    account_name: str = dspy.OutputField(desc="Account holder name")
    account_number: str = dspy.OutputField(
        desc="Account number or IBAN"
    )


## Runnable invoice fixtures


In [ ]:
invoice_text = """ACME Analytics LLC
Invoice #: A-1042
Invoice Date: 2026-06-30
Bill To: Northwind Research
Total Due: USD 1,250.00
Bank: First Example Bank
Account Name: ACME Analytics LLC
IBAN: GB82 WEST 1234 5698 7654 32
"""

train_examples = [
    dspy.Example(
        text=invoice_text,
        company="ACME Analytics LLC",
        billed_to="Northwind Research",
        invoice_number="A-1042",
        invoice_date="2026-06-30",
        total_amount="USD 1,250.00",
        bank_name="First Example Bank",
        account_name="ACME Analytics LLC",
        account_number="GB82 WEST 1234 5698 7654 32",
    ).with_inputs("text"),
    dspy.Example(
        text="Seller: Contoso Ltd | Inv# C-9 | Customer: Fabrikam | Date: 07/01/2026 | Amount Due: $88.40 | Bank: Example Credit Union | A/C Name: Contoso Ltd | A/C: 44332211",
        company="Contoso Ltd",
        billed_to="Fabrikam",
        invoice_number="C-9",
        invoice_date="07/01/2026",
        total_amount="$88.40",
        bank_name="Example Credit Union",
        account_name="Contoso Ltd",
        account_number="44332211",
    ).with_inputs("text"),
]


## Partial-credit metric


In [ ]:
FIELD_KEYS = [
    'company', 'billed_to', 'invoice_number', 'invoice_date',
    'total_amount', 'bank_name', 'account_name', 'account_number'
]

def field_accuracy_metric(gold, pred, trace=None,
                          pred_name=None, pred_trace=None):
    present = 0
    correct = 0

    for field in FIELD_KEYS:
        gold_value = getattr(gold, field)
        pred_value = getattr(pred, field, "")

        # 0.5 points for field presence (non-empty)
        if pred_value and pred_value.strip():
            present += 1

        # 0.5 points for field correctness
        if gold_value.strip().lower() == pred_value.strip().lower():
            correct += 1

    score = ((0.5 * present) + (0.5 * correct)) / len(FIELD_KEYS)
    return score


## Majority voting

This cell performs three live completions and votes independently on each field.


In [ ]:

def normalize_field(value):
    return str(value).strip().casefold()


extractor = dspy.ChainOfThought(
    InvoiceExtraction,
    n=3,
)
result = extractor(text=invoice_text)

field_votes = {}
for field in FIELD_KEYS:
    voted = dspy.majority(
        result,
        field=field,
        normalize=normalize_field,
    )
    field_votes[field] = getattr(voted, field)

majority_result = dspy.Prediction(**field_votes)


## GEPA instruction learning

This example uses the chapter's reflection model. It is intentionally tagged as expensive and requires provider access to that model.


In [ ]:
from dspy.teleprompt import GEPA

reflection_lm = dspy.LM('openai/gpt-5.6-sol', temperature=1.0)

optimizer = GEPA(
    metric=field_accuracy_metric,
    auto='light',
    reflection_lm=reflection_lm,
    num_threads=1
)

optimized_extractor = optimizer.compile(
    student=dspy.ChainOfThought(InvoiceExtraction),
    trainset=train_examples
)
